# Historical Modeling

Tujuan: Mengevaluasi pengaruh *historical features* terhadap performa model prediksi hasil pertandingan sepak bola internasional serta membandingkannya dengan *baseline model*.

## 1. Load Historical Dataset

Memuat dataset hasil *Historical Feature Engineering* yang akan digunakan untuk membangun dan mengevaluasi model.

In [7]:
import pandas as pd

In [8]:
historical_df = pd.read_csv("../data/processed/historical_features.csv", parse_dates=["date"])

In [9]:
historical_df.head()

,date,home_team,away_team,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,away_last10_winrate,home_avg_goals_scored,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form,match_result
0,1872-11-30,Scotland,England,Friendly,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D
1,1873-03-08,England,Scotland,Friendly,False,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,H
2,1874-03-07,Scotland,England,Friendly,False,0.000000,0.500000,0.000000,0.500000,1.000000,2.000000,2.000000,1.000000,-1.000000,1.000000,H
3,1875-03-06,England,Scotland,Friendly,False,0.333333,0.333333,0.333333,0.333333,1.666667,1.333333,1.333333,1.666667,0.333333,-0.333333,D
4,1876-03-04,Scotland,England,Friendly,False,0.250000,0.250000,0.250000,0.250000,1.500000,1.750000,1.750000,1.500000,-0.250000,0.250000,H


In [10]:
historical_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       49433 non-null  datetime64[us]
 1   home_team                  49433 non-null  str           
 2   away_team                  49433 non-null  str           
 3   tournament                 49433 non-null  str           
 4   neutral                    49433 non-null  bool          
 5   home_last5_winrate         49292 non-null  float64       
 6   away_last5_winrate         49238 non-null  float64       
 7   home_last10_winrate        49292 non-null  float64       
 8   away_last10_winrate        49238 non-null  float64       
 9   home_avg_goals_scored      49292 non-null  float64       
 10  away_avg_goals_scored      49238 non-null  float64       
 11  home_avg_goals_conceded    49292 non-null  float64       
 12  away_avg_goals_

In [11]:
historical_df.shape

(49433, 16)

Dataset berhasil dimuat dan berisi seluruh fitur baseline serta sepuluh *historical features* yang telah dibangun pada notebook sebelumnya. Seluruh fitur tersebut merepresentasikan kondisi masing-masing tim **sebelum pertandingan berlangsung**, sehingga dapat digunakan sebagai masukan model tanpa menimbulkan *future data leakage*.

Pada tahap berikutnya, fitur (*X*) dan target (*y*) akan dipisahkan sebagai persiapan proses pelatihan model.

## 2. Feature & Target Separation
Sebelum membangun model, dataset dipisahkan menjadi **fitur (X)** dan **target (y)**. Variabel target yang akan diprediksi adalah `match_result`, sedangkan seluruh kolom lainnya yang relevan digunakan sebagai fitur prediktor. Kolom `date` tidak digunakan dalam proses pelatihan model karena hanya berfungsi sebagai acuan untuk melakukan *time-based train-test split*.

In [12]:
feature_columns = [
    "home_team",
    "away_team",
    "tournament",
    "neutral",

    "home_last5_winrate",
    "away_last5_winrate",

    "home_last10_winrate",
    "away_last10_winrate",

    "home_avg_goals_scored",
    "away_avg_goals_scored",

    "home_avg_goals_conceded",
    "away_avg_goals_conceded",

    "home_goal_difference_form",
    "away_goal_difference_form"
]

X = historical_df[feature_columns]
y = historical_df["match_result"]

In [13]:
# VERIFIKASI
print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (49433, 14)
Target Shape  : (49433,)


Pemisahan fitur dan target berhasil dilakukan. Model akan menggunakan **14 fitur**, yang terdiri dari:
- 4 fitur dasar (*baseline features*),
- 10 fitur historis (*historical features*).
Variabel target tetap berupa `match_result` dengan tiga kelas (`H`, `D`, dan `A`). Sementara itu, kolom `date` sengaja tidak disertakan sebagai fitur karena hanya digunakan untuk menjaga urutan kronologis saat proses *train-test split*.

## 3. Time-Based Train-Test Split

Dataset dibagi menjadi data latih (*training set*) dan data uji (*test set*) berdasarkan urutan waktu (*time-based split*). Berbeda dengan *random split*, pendekatan ini mempertahankan urutan kronologis pertandingan sehingga model hanya belajar dari pertandingan yang terjadi di masa lalu untuk memprediksi pertandingan di masa depan. Metode ini lebih mencerminkan kondisi prediksi di dunia nyata (*real-world prediction*) serta menghindari *future data leakage*. Untuk menjaga konsistensi eksperimen, proporsi pembagian data tetap sama seperti pada *Baseline Modeling*, yaitu **80% data latih** dan **20% data uji**.

In [14]:
split_index = int(len(historical_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [15]:
# VALIDASI
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

Training samples : 39,546
Testing samples  : 9,887


In [18]:
print("Training period")
print(
    historical_df.iloc[0]["date"],
    "---",
    historical_df.iloc[split_index - 1]["date"]
)

print()

print("Testing period")
print(
    historical_df.iloc[split_index]["date"],
    "---",
    historical_df.iloc[-1]["date"]
)

Training period
1872-11-30 00:00:00 --- 2016-03-25 00:00:00

Testing period
2016-03-25 00:00:00 --- 2026-06-18 00:00:00


Dataset berhasil dibagi menjadi data latih dan data uji menggunakan pendekatan *time-based split*. Seluruh pertandingan pada data latih terjadi lebih awal dibandingkan data uji, sehingga model hanya memanfaatkan informasi historis yang tersedia sebelum periode evaluasi. Dengan mempertahankan strategi pembagian data yang sama seperti pada *Baseline Modeling*, hasil evaluasi pada notebook ini dapat dibandingkan secara langsung untuk mengukur pengaruh penambahan *historical features* terhadap performa model.

## 4. Historical Modeling

Pada tahap ini akan dibangun dua model klasifikasi menggunakan dataset yang telah diperkaya dengan *historical features*. Proses pemodelan tetap mengikuti pendekatan yang digunakan pada *Baseline Modeling* agar hasil evaluasi dapat dibandingkan secara adil. Perbedaannya hanya terletak pada informasi yang diberikan kepada model, yaitu penambahan fitur historis yang merepresentasikan performa masing-masing tim sebelum pertandingan berlangsung.

### 4.1 Feature Types
Dataset terdiri atas fitur kategorikal dan numerik sehingga masing-masing memerlukan perlakuan yang berbeda pada tahap *preprocessing*. Fitur kategorikal akan diubah menjadi representasi numerik menggunakan **One-Hot Encoding**, sedangkan fitur numerik akan diteruskan (*passthrough*) tanpa transformasi tambahan.

In [ ]:
# KATEGORIKAL
categorical_features = [
    "home_team",
    "away_team",
    "tournament"
]

# NUMERIK
numerical_features = [
    "neutral",

    "home_last5_winrate",
    "away_last5_winrate",

    "home_last10_winrate",
    "away_last10_winrate",

    "home_avg_goals_scored",
    "away_avg_goals_scored",

    "home_avg_goals_conceded",
    "away_avg_goals_conceded",

    "home_goal_difference_form",
    "away_goal_difference_form"
]

# %%
print("Categorical Features :", len(categorical_features))
print("Numerical Features   :", len(numerical_features))
print("Total Features       :", len(categorical_features) + len(numerical_features))

Seluruh fitur berhasil dikelompokkan berdasarkan tipe datanya. Terdapat **3 fitur kategorikal** (`home_team`, `away_team`, dan `tournament`) yang akan diproses menggunakan *One-Hot Encoding*. Sementara itu, **11 fitur numerik** terdiri atas satu fitur boolean (`neutral`) dan sepuluh *historical features* yang akan diteruskan langsung ke model tanpa transformasi tambahan. Pemisahan ini memungkinkan seluruh proses *preprocessing* dilakukan secara otomatis melalui *Pipeline* pada tahap berikutnya.